In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)


In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# 4. Print shape of one batch

print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here: with an activation function right?
import torch.nn as nn

class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim):
        super(NN4Layer, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)

        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        self.layer4 = nn.Linear(hidden_dim, hidden_dim)
        # activation function for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # Layer 1:
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        # Layer 2
        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        # Layer 3
        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        # Layer 4
        z4 = self.layer4(a3)

        return z4

In [ ]:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  model.train()

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    X_batch = X_batch.view(X_batch.shape[0], -1).to(device)
    y_batch = y_batch.view(-1,1).to(device)

    # Forward Pass
    outputs = model(X_batch)
    loss = criterion(outputs, y_batch)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
def validate(model, criterion, test_loader, device):
  model.eval()

  running_loss = 0.0
  device = 'cpu'
  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      #Move data to device

      X_batch = X_batch.view(X_batch.shape[0], -1).to(device)
      y_batch = y_batch.view(-1,1).to(device)

      # Forward pass
      outputs = model(X_batch)
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Define the device, model, loss function, and optimizer
device = 'cpu'
# Model parameters
print(X_train.shape)
input_dim = 3*36*36
hidden_dim = 1000


model = NN4Layer(input_dim, hidden_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Define the device, model, loss function, and optimizer part 2
from torch.optim import AdamW
lr = 0.001

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=lr)

In [ ]:
# Run Training
train_losses = []
val_losses = []
num_epochs = 20
print('Starting Training...')
for epoch in range(num_epochs):
  # training 1 epoch
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # validation
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
# Set model to evaluation mode
model.eval()
# Get one batch from the test DataLoader
images, labels = next(iter(test_loader))
# Move images to device
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    # Flatten images before passing to the model
    outputs = model(images.view(images.size(0), -1)) # batch size i think yea!
    predictions = torch.argmax(outputs, dim=1)

# Move tensors back to CPU for plotting
images = images.cpu()
print(images.shape)
labels = labels.cpu()
predictions = predictions.cpu()

# Plot first 6 predictions
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].view(36, 36, 3))  # I THINK I NEED TO RE-ORDER THE SHAPE BUT HOW!!!!!!!!
    plt.title(f"True: {labels[i]} | Pred: {predictions[i]}")
    plt.axis('off')

plt.tight_layout()
plt.show()